# "A note on Cross Entropy loss with PyTorch"

- toc: true
- badges: true
- comments: true
- categories: [fastai, pytorch, jupyter]

In [1]:
from fastai.vision.all import *

Pretend the following are the activations of each class of a multiclass classification problem. So we have 6 examples and in each row we have the activation for each class the example could belong to.

In [2]:
activations = torch.randn((6,2))*2
activations

tensor([[ 2.5256, -0.4387],
        [-0.1350,  0.0298],
        [-0.3085, -1.3541],
        [-0.5117,  1.4399],
        [ 0.1849, -3.1652],
        [ 0.5503, -0.4122]])

Suppose the correct class of each example is as follows

In [3]:
targets = tensor([0,1,0,1,1,0])
targets

tensor([0, 1, 0, 1, 1, 0])

Take the softmax of the activations

In [4]:
sm_acts = torch.softmax(activations, dim=1)
sm_acts

tensor([[0.9509, 0.0491],
        [0.4589, 0.5411],
        [0.7399, 0.2601],
        [0.1244, 0.8756],
        [0.9661, 0.0339],
        [0.7236, 0.2764]])

Extract the probabilities predicted for the correct class.

In [5]:
idx = range(6)
list(idx)

[0, 1, 2, 3, 4, 5]

In [6]:
p_correct_class = sm_acts[idx, targets]
p_correct_class

tensor([0.9509, 0.5411, 0.7399, 0.8756, 0.0339, 0.7236])

Take the log of the softmax activations

In [7]:
torch.log(sm_acts)

tensor([[-0.0503, -3.0146],
        [-0.7790, -0.6141],
        [-0.3012, -1.3468],
        [-2.0845, -0.1328],
        [-0.0345, -3.3846],
        [-0.3235, -1.2860]])

Computing the softmax of the activations and then taking the log is equivalent to applying PyTorch's log_softmax function directly to the original activations. We want to do the latter because it will faster and more accurate.

In [8]:
torch.log_softmax(activations, dim=1)

tensor([[-0.0503, -3.0146],
        [-0.7790, -0.6141],
        [-0.3012, -1.3468],
        [-2.0845, -0.1328],
        [-0.0345, -3.3846],
        [-0.3235, -1.2860]])

Suppose we have $N$ training examples and $p_{i}$ gives the probability of the correct class for the $i$-th training example. We want to maximize: $$\prod_{i=1}^{N} p_i$$ 

This is equivalent to maximizing $$ln(\prod_{i=1}^{N}p_i) = \sum_{i=1}^{N}ln(p_{i})$$ 

Which is the same as minimizing the sum of the negative log likelihoods $$-\sum_{i=1}^{N}ln(p_{i})$$

The above can now serve as a loss function.

Recall Cross Entropy = $-\sum_{k=1}^{K}q_{k}ln(p_{k})$ where $q$ is the reference distribution over $K$ classes while our predictions over the $K$ classes have the distribution $p$. Observe this becomes just a single term when, in the reference distribution $q$, only one of the classes has a probability of $1$.

Thus, $-\sum_{i=1}^{N}ln(p_{i})$ can be interpreted as the sum of cross entropy losses across all examples.

Next we can compute the above quantity as follows:

In [9]:
-1*torch.log(p_correct_class), (-1*torch.log(p_correct_class)).mean()

(tensor([0.0503, 0.6141, 0.3012, 0.1328, 3.3846, 0.3235]), tensor(0.8011))

We can use Pytorch to compute this directly

In [10]:
nn.CrossEntropyLoss(reduction='none')(activations, targets), nn.CrossEntropyLoss()(activations, targets)

(tensor([0.0503, 0.6141, 0.3012, 0.1328, 3.3846, 0.3235]), tensor(0.8011))

or by using:

In [11]:
F.cross_entropy(activations, targets, reduction='none'), F.cross_entropy(activations, targets)

(tensor([0.0503, 0.6141, 0.3012, 0.1328, 3.3846, 0.3235]), tensor(0.8011))

# Gradient of Cross Entropy

We follow the note in {% cite markusthill_ce_note %}

# References
{% bibliography --cited %}